# Agents with a Framework: Pydantic AI

In `agent_tutorial.ipynb` we wrote the agent loop by hand in about 20 lines. That is the right way to
learn it. It is not the way you want to maintain it: the moment you need validated outputs, several tools
sharing state, conversation memory, retries, or swapping models, the hand-written loop grows fast.

**Pydantic AI** is an agent framework from the team behind Pydantic, the data-validation library used by
FastAPI and most of the Python ML stack. Its pitch is simple: *an agent is a typed Python object*. Tools
are typed functions, outputs are Pydantic models, and the framework runs the loop.

This notebook rebuilds the same agent as before, piece by piece, and points out what the framework is
doing for you at each step. Same local model, same Ollama, no API keys.

| Notebook 1 (by hand) | This notebook (Pydantic AI) |
|---|---|
| `messages` list you manage | `message_history=` |
| `tools=[fn, fn]` + name registry | `@agent.tool` decorators |
| `while` loop with `max_steps` | `await agent.run(...)` |
| final answer is a string | final answer is a **validated Python object** |
| globals for shared state | typed **dependencies** passed into tools |

## 0. Setup

Pydantic AI talks to Ollama through Ollama's OpenAI-compatible endpoint, so the model is just a string
`"ollama:<model>"` once `OLLAMA_BASE_URL` is set.

One habit to pick up: Pydantic AI is **async-first**. Jupyter lets you write `await` at the top level
of a cell, so every run below looks like `result = await agent.run(...)`. In a plain script you would use
`agent.run_sync(...)` instead.

In [1]:
import os
os.environ["OLLAMA_BASE_URL"] = "http://localhost:11434/v1"
os.environ["PYDANTIC_AI_NO_BANNER"] = "1"

from pydantic_ai import Agent

MODEL = "ollama:gemma4:26b"   # same model as the first notebook

## 1. The smallest possible agent

An `Agent` is created once and reused. `instructions` is the system prompt.
`result.output` is the answer.

In [2]:
agent = Agent(MODEL, instructions="You are a concise assistant. Answer in one or two sentences.")

result = await agent.run("What is an AI agent?")
print(result.output)

An AI agent is an autonomous system that uses artificial intelligence to perceive its environment, reason through tasks, and take independent actions to achieve specific goals. It is distinguished from a standard chatbot by its ability to use external tools and interact with software to complete complex workflows without constant human intervention.


## 2. Tools, and proof that the loop is still there

Tools are the same typed, docstringed functions as before, registered with a decorator.
`@agent.tool_plain` is for tools that need nothing from the agent except their arguments.

In [3]:
from datetime import date

@agent.tool_plain
def calculate(expression: str) -> str:
    """Evaluate a math expression and return the result, e.g. "2 ** 10" or "(3.5 + 1) * 4"."""
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Error: {e}"

@agent.tool_plain
def get_today() -> str:
    """Return today's date in YYYY-MM-DD format."""
    return date.today().isoformat()

result = await agent.run("What is 2 to the power 20, divided by 3? Two decimal places.")
print(result.output)

2 to the power 20, divided by 3, is 349,525.33.


Nothing printed a trace this time, so where did the loop go? It ran inside `agent.run`.
`result.all_messages()` is the full transcript. Each message is made of **parts**, and the part types
spell out exactly the loop you wrote by hand: prompt, tool call, tool return, final text.

You may also see `ThinkingPart` entries. gemma4 is a reasoning model and Ollama exposes its private reasoning; Pydantic AI keeps it in the transcript. It is not part of the answer.

In [4]:
for message in result.all_messages():
    for part in message.parts:
        kind = type(part).__name__
        detail = getattr(part, "content", None) or getattr(part, "args", None) or ""
        print(f"{kind:16} {str(detail)[:90]}")

UserPromptPart   What is 2 to the power 20, divided by 3? Two decimal places.
ThinkingPart     The user wants to calculate $2^{20} / 3$ and wants the result rounded to two decimal place
ToolCallPart     {"expression":"2**20 / 3"}
ToolReturnPart   349525.3333333333
ThinkingPart     The calculation result is 349525.3333333333.
The user asked for two decimal places.
The re
TextPart         2 to the power 20, divided by 3, is 349,525.33.


## 3. Structured output: the big win

By hand, the agent's final answer was a string you had to parse. With Pydantic AI you declare the
**shape** of the answer as a Pydantic model and get a validated Python object back. If the model
returns something that does not fit, the framework sends the validation error back to the model
and asks it to try again (`retries`, default 1).

This is the single most useful feature for research pipelines: agent output becomes data you can
put straight into a DataFrame.

In [5]:
from pydantic import BaseModel, Field

class Expense(BaseModel):
    item: str
    amount: float = Field(gt=0, description="Amount in dollars")

class ExpenseReport(BaseModel):
    expenses: list[Expense]
    total: float = Field(description="Sum of all amounts")
    most_expensive: str = Field(description="Item with the largest amount")

extractor = Agent(MODEL, output_type=ExpenseReport, instructions="Extract the expenses from the text.")

result = await extractor.run("Today I spent: coffee 4.50, lunch 12.25, train 7.80, book 18.99")
report = result.output

print(type(report).__name__)
print(report.total, report.most_expensive)
print(report.model_dump_json(indent=2))

ExpenseReport
43.54 book
{
  "expenses": [
    {
      "item": "coffee",
      "amount": 4.5
    },
    {
      "item": "lunch",
      "amount": 12.25
    },
    {
      "item": "train",
      "amount": 7.8
    },
    {
      "item": "book",
      "amount": 18.99
    }
  ],
  "total": 43.54,
  "most_expensive": "book"
}


## 4. Dependencies: shared state without globals

In notebook 1 the file tools reached for a global `WORKSPACE` path. Pydantic AI instead lets you
declare a **dependencies type** and pass an instance into each run. Tools that need it take a
`RunContext` first argument and read `ctx.deps`.

Why bother? You can point the same agent at a different folder (or database, or API client) per
run, and in tests you can pass a fake. Nothing is hidden in module state.

In [6]:
from dataclasses import dataclass
from pathlib import Path
from pydantic_ai import RunContext

@dataclass
class Deps:
    workspace: Path

files_agent = Agent(
    MODEL,
    deps_type=Deps,
    instructions="You are a helpful assistant with access to a workspace folder. Use tools when they help.",
)

@files_agent.tool
def write_file(ctx: RunContext[Deps], name: str, content: str) -> str:
    """Write text to a file in the workspace (overwrites)."""
    (ctx.deps.workspace / name).write_text(content)
    return f"Wrote {len(content)} characters to {name}"

@files_agent.tool
def read_file(ctx: RunContext[Deps], name: str) -> str:
    """Read and return the contents of a file in the workspace."""
    path = ctx.deps.workspace / name
    return path.read_text() if path.exists() else f"Error: {name} does not exist"

@files_agent.tool
def list_files(ctx: RunContext[Deps]) -> str:
    """List the files in the workspace."""
    return ", ".join(p.name for p in ctx.deps.workspace.iterdir()) or "(empty)"

@files_agent.tool_plain
def calculate(expression: str) -> str:
    """Evaluate a math expression and return the result, e.g. "4.5 + 12.25"."""
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Error: {e}"

deps = Deps(workspace=Path("workspace"))
deps.workspace.mkdir(exist_ok=True)

result = await files_agent.run("What files are in the workspace?", deps=deps)
print(result.output)

The workspace contains the following files:
- `expenses.txt`
- `total.txt`


## 5. The multi-step task, with a typed result

Same task as the end of notebook 1, but now the agent must return a `TaskResult` object. Note the
`output_type=` can be set per run, so one agent can serve several result shapes.

In [7]:
class TaskResult(BaseModel):
    total: float = Field(description="The computed total")
    file_written: str = Field(description="Name of the file the total was written to")
    steps_taken: list[str] = Field(description="Short description of each tool call made, in order")

(deps.workspace / "expenses.txt").write_text("coffee 4.50\nlunch 12.25\ntrain 7.80\nbook 18.99")
(deps.workspace / "total.txt").unlink(missing_ok=True)

task = (
    "Read expenses.txt, add up all the amounts with the calculator, "
    "and write the total to total.txt as 'Total: <amount>'."
)
result = await files_agent.run(task, deps=deps, output_type=TaskResult)

print(result.output.model_dump_json(indent=2))
print("\ntotal.txt on disk:", (deps.workspace / "total.txt").read_text())

{
  "total": 43.54,
  "file_written": "total.txt",
  "steps_taken": [
    "List files to confirm expenses.txt exists",
    "Read expenses.txt to extract amounts",
    "Calculate the sum of the extracted amounts (4.50 + 12.25 + 7.80 + 18.99)",
    "Write the final total to total.txt"
  ]
}

total.txt on disk: Total: 43.54


Look at the transcript again. Every tool call the framework made is there, plus a final
`ToolCallPart` named `final_result`: that is how Pydantic AI asks the model for structured output,
by turning your `TaskResult` schema into one more tool.

In [8]:
for message in result.all_messages():
    for part in message.parts:
        if type(part).__name__ in ("ToolCallPart", "ToolReturnPart"):
            print(f"{type(part).__name__:15} {part.tool_name:14} {str(getattr(part, 'args', None) or part.content)[:70]}")

ToolCallPart    list_files     {}
ToolReturnPart  list_files     expenses.txt
ToolCallPart    read_file      {"name":"expenses.txt"}
ToolReturnPart  read_file      coffee 4.50
lunch 12.25
train 7.80
book 18.99
ToolCallPart    calculate      {"expression":"4.50 + 12.25 + 7.80 + 18.99"}
ToolReturnPart  calculate      43.54
ToolCallPart    write_file     {"content":"Total: 43.54","name":"total.txt"}
ToolReturnPart  write_file     Wrote 12 characters to total.txt
ToolCallPart    final_result   {"file_written":"total.txt","steps_taken":["List files to confirm expe
ToolReturnPart  final_result   Final result processed.


## 6. Memory across runs

Each `run` is stateless, exactly like notebook 1. To continue a conversation, pass the previous
messages back in. `new_messages()` returns only what the last run added.

In [9]:
first = await agent.run("My name is Nick and I'm preparing a lecture on tokens.")
second = await agent.run("What am I preparing?", message_history=first.new_messages())
print(second.output)

You are preparing a lecture on tokens.


## 7. Counting what it cost

Every result carries token usage. These are the tokens from `tokens_tutorial.ipynb`: this is the
number you would be billed for with a cloud model, and the amount of context window consumed.

In [10]:
u = result.usage
print(f"requests: {u.requests}   input tokens: {u.input_tokens}   output tokens: {u.output_tokens}")

requests: 5   input tokens: 3040   output tokens: 478


## 8. What the framework bought you

Everything in this notebook is the same loop from notebook 1. What changed:

- **Typed outputs.** `output_type=SomeModel` turns free text into validated data, with automatic
  retry on bad output. For a research pipeline this is the feature that matters.
- **Dependencies.** `deps_type` replaces globals; tools get `ctx.deps`. Easy to test, easy to swap.
- **Transcript and usage** for free: `all_messages()`, `usage`.
- **Model-agnostic.** Change `MODEL` to `"anthropic:claude-opus-5"` or `"openai:gpt-5"` and
  nothing else changes. (Those need API keys; everything here ran locally.)

**Things to try next:**
1. Set `retries=3` on `extractor` and feed it messy text. Watch validation drive the retries.
2. Give `TaskResult` a field the model cannot know, e.g. `confidence: float`, and see what it does.
3. Make `read_file` raise an exception for a missing file instead of returning an error string.
   Pydantic AI will report it to the model as a tool error; compare how the model recovers.
4. Stream the answer: `async with agent.run_stream(...) as stream: async for text in stream.stream_text(): ...`
5. Switch `MODEL` to `"ollama:qwen2.5:7b"` and see which features a small model struggles with.